# EXHEART Notebook 07 - full-stack cross-fitted calibration and editor revision rerun

This notebook addresses the Editor-in-Chief's calibration concern using **two-stage full-stack cross-fitting**.

For every calibration score:

1. The observation is excluded from the base learners that generate its four base-model probabilities.
2. The same observation is excluded from the logistic meta-learner that combines those probabilities.
3. Platt scaling is fitted only after all observations have an out-of-sample score from the **complete stack**.
4. The locked 20% test set is never used for base fitting, meta fitting, calibration fitting, threshold selection, or mitigation fitting.

The notebook reruns the four analysis arms required by the manuscript:

- BRFSS 2015 primary model
- BRFSS 2015 -> BRFSS 2020 naive transport
- BRFSS 2020 independently retrained model
- Cardio secondary examination-cohort benchmark
- BRFSS shared-feature harmonised transport

It also recomputes threshold-dependent fairness and selects the female equal-opportunity threshold on cross-fitted development predictions, not on the test set.

**Mandatory final run:** use a Colab GPU runtime and set `FULL_RUN = True`. The full run can take 1-3 hours depending on the runtime.

In [1]:
# Install only when needed in Colab.
!pip -q install lightgbm xgboost joblib python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 6.5 MB/s eta 0:00:00


In [2]:
# Determinism and imports - run before importing TensorFlow.
import os, json, random, hashlib, warnings, shutil
os.environ['PYTHONHASHSEED'] = '42'
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    confusion_matrix, roc_curve, precision_recall_curve
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

TensorFlow: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
# Paths. The notebook first tries Google Drive, then a local Colab clone.
from google.colab import drive
drive.mount('/content/drive')

DRIVE_REPO = '/content/drive/MyDrive/EXHEART_Research/exheart-research'
LOCAL_REPO = '/content/exheart-research'
REPO_URL = 'https://github.com/anasbiswas1/exheart-research.git'

if os.path.exists(DRIVE_REPO):
    REPO = DRIVE_REPO
else:
    if not os.path.exists(LOCAL_REPO):
        import subprocess
        subprocess.run(['git','clone','--depth','1',REPO_URL,LOCAL_REPO], check=True)
    REPO = LOCAL_REPO

OUT = os.path.join(REPO, 'results', 'editor_revision_fullstack_crossfit')
MODEL_OUT = os.path.join(REPO, 'models', 'editor_revision_fullstack_crossfit')
FIG_OUT = os.path.join(OUT, 'figures')
os.makedirs(OUT, exist_ok=True); os.makedirs(MODEL_OUT, exist_ok=True); os.makedirs(FIG_OUT, exist_ok=True)

DATA15 = os.path.join(REPO, 'data/brfss2015/heart_disease_health_indicators_BRFSS2015.csv')
DATA20 = os.path.join(REPO, 'data/brfss2020/heart_2020_cleaned.csv')
DATAC = os.path.join(REPO, 'data/cardio/cardio_train.csv')

for p in [DATA15, DATA20, DATAC]:
    assert os.path.exists(p), f'Missing data file: {p}'
print('Repository:', REPO)
print('Output:', OUT)

Mounted at /content/drive
Repository: /content/drive/MyDrive/EXHEART_Research/exheart-research
Output: /content/drive/MyDrive/EXHEART_Research/exheart-research/results/editor_revision_fullstack_crossfit


## Configuration

`FULL_RUN=True` is required for submission results. `FAST_SMOKE_TEST=True` draws a stratified subset only to test that the notebook runs; never use smoke-test outputs in the manuscript.

In [4]:
FULL_RUN = True
FAST_SMOKE_TEST = False
N_BASE_FOLDS = 5
N_META_FOLDS = 5
EPOCHS = 50
BATCH_SIZE = 512

assert not (FULL_RUN and FAST_SMOKE_TEST)

In [5]:
# Core metrics and model builders.
def compute_ece(y_true, y_prob, n_bins=10):
    y_true = np.asarray(y_true); y_prob = np.asarray(y_prob)
    bins = np.linspace(0, 1, n_bins + 1)
    total = 0.0
    for i in range(n_bins):
        if i == n_bins - 1:
            m = (y_prob >= bins[i]) & (y_prob <= bins[i+1])
        else:
            m = (y_prob >= bins[i]) & (y_prob < bins[i+1])
        if m.any():
            total += m.sum() * abs(y_true[m].mean() - y_prob[m].mean())
    return total / len(y_true)

def build_mlp(d):
    inp = keras.Input(shape=(d,))
    x = layers.Dense(256, activation='relu')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    model = keras.Model(inp, out)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=[keras.metrics.AUC(name='auc')]
    )
    return model

def callbacks():
    return [
        keras.callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=5, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor='val_auc', mode='max', factor=0.5, patience=3, min_lr=1e-6)
    ]

def class_weights(y):
    classes = np.array([0, 1])
    w = compute_class_weight('balanced', classes=classes, y=np.asarray(y).astype(int))
    return {0: float(w[0]), 1: float(w[1])}, float(w[1] / w[0])

def make_tree_models(spw, seed):
    return (
        XGBClassifier(
            n_estimators=300, max_depth=6, learning_rate=0.05,
            scale_pos_weight=spw, eval_metric='logloss',
            random_state=seed, n_jobs=-1
        ),
        LGBMClassifier(
            n_estimators=300, max_depth=6, learning_rate=0.05,
            class_weight='balanced', random_state=seed, n_jobs=-1, verbose=-1
        ),
        RandomForestClassifier(
            n_estimators=200, max_depth=10, class_weight='balanced',
            random_state=seed, n_jobs=-1
        )
    )

def metrics_at_threshold(y, prob, threshold):
    y = np.asarray(y).astype(int); prob = np.asarray(prob)
    pred = (prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0,1]).ravel()
    return {
        'AUC': float(roc_auc_score(y, prob)),
        'AUPRC': float(average_precision_score(y, prob)),
        'Brier': float(brier_score_loss(y, prob)),
        'ECE': float(compute_ece(y, prob)),
        'Sensitivity': float(tp / (tp + fn)) if (tp + fn) else np.nan,
        'Specificity': float(tn / (tn + fp)) if (tn + fp) else np.nan,
        'FPR': float(fp / (fp + tn)) if (fp + tn) else np.nan,
        'PPV': float(tp / (tp + fp)) if (tp + fp) else np.nan,
        'NPV': float(tn / (tn + fn)) if (tn + fn) else np.nan,
        'SelectionRate': float(pred.mean()),
        'BalancedAccuracy': float(0.5*((tp/(tp+fn)) + (tn/(tn+fp)))) if (tp+fn and tn+fp) else np.nan,
        'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp),
        'threshold': float(threshold)
    }

def net_benefit(y, prob, threshold):
    y = np.asarray(y).astype(int); pred = (np.asarray(prob) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0,1]).ravel()
    n = len(y)
    return (tp/n) - (fp/n)*(threshold/(1-threshold))

def bootstrap_auc_ci(y, prob, B=1000, seed=SEED):
    rng=np.random.default_rng(seed); y=np.asarray(y); prob=np.asarray(prob); vals=[]
    for _ in range(B):
        idx=rng.integers(0,len(y),len(y))
        if len(np.unique(y[idx]))<2: continue
        vals.append(roc_auc_score(y[idx],prob[idx]))
    return np.quantile(vals,[0.025,0.975]).tolist()

def bootstrap_binary_gap_ci(y, prob, group, threshold, values=(0,1), B=1000, seed=SEED):
    rng=np.random.default_rng(seed); y=np.asarray(y); prob=np.asarray(prob); group=np.asarray(group); vals=[]
    for _ in range(B):
        idx=rng.integers(0,len(y),len(y)); tprs=[]
        for v in values:
            m=(group[idx]==v)&(y[idx]==1)
            if m.sum()==0: break
            tprs.append((prob[idx][m]>=threshold).mean())
        if len(tprs)==len(values): vals.append(max(tprs)-min(tprs))
    return np.quantile(vals,[0.025,0.975]).tolist()


## Full-stack cross-fitting

The first cross-fitting layer creates OOF predictions for the four base learners. The second cross-fitting layer fits a logistic meta-learner on four-fifths of the OOF matrix and predicts the remaining fifth. Platt scaling is fitted on those second-stage held-out scores.

In [6]:
def fit_fullstack_crossfitted(X, y, threshold, arm, group_df=None, seed=SEED):
    X = pd.DataFrame(X).reset_index(drop=True)
    y = pd.Series(y).astype(int).reset_index(drop=True)
    if FAST_SMOKE_TEST:
        idx, _ = train_test_split(np.arange(len(y)), train_size=min(20000, len(y)), stratify=y, random_state=seed)
        X = X.iloc[idx].reset_index(drop=True); y = y.iloc[idx].reset_index(drop=True)
        if group_df is not None: group_df = group_df.iloc[idx].reset_index(drop=True)

    idx_all = np.arange(len(y))
    idx_train, idx_test = train_test_split(idx_all, test_size=0.20, stratify=y, random_state=seed)
    Xtr, Xte = X.iloc[idx_train].reset_index(drop=True), X.iloc[idx_test].reset_index(drop=True)
    ytr, yte = y.iloc[idx_train].reset_index(drop=True), y.iloc[idx_test].reset_index(drop=True)
    gte = group_df.iloc[idx_test].reset_index(drop=True) if group_df is not None else None

    Xtr_np, Xte_np = Xtr.to_numpy(dtype=np.float32), Xte.to_numpy(dtype=np.float32)
    ntr, d = Xtr_np.shape
    cw, spw = class_weights(ytr)

    # Stage 1: base-model OOF predictions.
    oof_base = np.full((ntr, 4), np.nan, dtype=float)
    base_seen = np.zeros(ntr, dtype=int)
    base_cv = StratifiedKFold(n_splits=N_BASE_FOLDS, shuffle=True, random_state=seed)

    for fold, (itr, iva) in enumerate(base_cv.split(Xtr_np, ytr), start=1):
        # Every fold-specific modelling choice is estimated only from the in-fold data.
        cw_fold, spw_fold = class_weights(ytr.iloc[itr])
        xa, lg, rf = make_tree_models(spw_fold, seed + fold)
        xa.fit(Xtr_np[itr], ytr.iloc[itr]); lg.fit(Xtr_np[itr], ytr.iloc[itr]); rf.fit(Xtr_np[itr], ytr.iloc[itr])

        fold_scaler = StandardScaler().fit(Xtr_np[itr])
        Xfit_sc = fold_scaler.transform(Xtr_np[itr]); Xval_sc = fold_scaler.transform(Xtr_np[iva])
        tf.keras.backend.clear_session(); tf.keras.utils.set_random_seed(seed + fold)
        mlp = build_mlp(d)
        mlp.fit(
            Xfit_sc, ytr.iloc[itr].to_numpy(), epochs=EPOCHS, batch_size=BATCH_SIZE,
            validation_split=0.10, class_weight=cw_fold, callbacks=callbacks(), verbose=0
        )

        oof_base[iva, 0] = xa.predict_proba(Xtr_np[iva])[:, 1]
        oof_base[iva, 1] = lg.predict_proba(Xtr_np[iva])[:, 1]
        oof_base[iva, 2] = rf.predict_proba(Xtr_np[iva])[:, 1]
        oof_base[iva, 3] = mlp.predict(Xval_sc, verbose=0).ravel()
        base_seen[iva] += 1
        print(f'{arm}: base OOF fold {fold}/{N_BASE_FOLDS}')

    assert np.all(base_seen == 1), 'Each development observation must receive one base-OOF prediction.'
    assert np.isfinite(oof_base).all(), 'Non-finite base OOF predictions.'

    # Stage 2: cross-fitted meta-learner predictions.
    meta_cf_raw = np.full(ntr, np.nan, dtype=float)
    meta_seen = np.zeros(ntr, dtype=int)
    meta_cv = StratifiedKFold(n_splits=N_META_FOLDS, shuffle=True, random_state=seed + 1000)
    for fold, (itr, iva) in enumerate(meta_cv.split(oof_base, ytr), start=1):
        meta_fold = LogisticRegression(class_weight='balanced', max_iter=2000, random_state=seed + fold)
        meta_fold.fit(oof_base[itr], ytr.iloc[itr])
        meta_cf_raw[iva] = meta_fold.predict_proba(oof_base[iva])[:, 1]
        meta_seen[iva] += 1
        print(f'{arm}: meta OOF fold {fold}/{N_META_FOLDS}')

    assert np.all(meta_seen == 1), 'Each development observation must receive one meta-held-out prediction.'
    assert np.isfinite(meta_cf_raw).all(), 'Non-finite cross-fitted stack predictions.'

    # Honest calibrator and final meta-learner.
    platt = LogisticRegression(max_iter=2000, random_state=seed)
    platt.fit(meta_cf_raw.reshape(-1, 1), ytr)
    meta_final = LogisticRegression(class_weight='balanced', max_iter=2000, random_state=seed)
    meta_final.fit(oof_base, ytr)

    # Final base learners fitted only on development data.
    xgb, lgbm, rf = make_tree_models(spw, seed)
    xgb.fit(Xtr_np, ytr); lgbm.fit(Xtr_np, ytr); rf.fit(Xtr_np, ytr)
    final_scaler = StandardScaler().fit(Xtr_np)
    tf.keras.backend.clear_session(); tf.keras.utils.set_random_seed(seed)
    mlp_final = build_mlp(d)
    mlp_final.fit(
        final_scaler.transform(Xtr_np), ytr.to_numpy(), epochs=EPOCHS, batch_size=BATCH_SIZE,
        validation_split=0.10, class_weight=cw, callbacks=callbacks(), verbose=0
    )

    base_test = np.column_stack([
        xgb.predict_proba(Xte_np)[:, 1], lgbm.predict_proba(Xte_np)[:, 1],
        rf.predict_proba(Xte_np)[:, 1], mlp_final.predict(final_scaler.transform(Xte_np), verbose=0).ravel()
    ])
    raw_test = meta_final.predict_proba(base_test)[:, 1]
    prob_test = platt.predict_proba(raw_test.reshape(-1, 1))[:, 1]

    # Cross-fitted calibrated development scores are used for mitigation fitting only.
    prob_dev_cf = platt.predict_proba(meta_cf_raw.reshape(-1, 1))[:, 1]

    result = {
        'arm': arm, 'threshold': threshold,
        'X_columns': X.columns.tolist(),
        'train_indices': idx_train, 'test_indices': idx_test,
        'X_train': Xtr, 'X_test': Xte, 'y_train': ytr, 'y_test': yte,
        'groups_train': group_df.iloc[idx_train].reset_index(drop=True) if group_df is not None else None,
        'groups_test': gte,
        'base_oof': oof_base, 'meta_cf_raw': meta_cf_raw, 'prob_dev_cf': prob_dev_cf,
        'raw_test': raw_test, 'prob_test': prob_test,
        'metrics_raw': metrics_at_threshold(yte, raw_test, threshold),
        'metrics_calibrated': metrics_at_threshold(yte, prob_test, threshold),
        'models': {'xgb': xgb, 'lgbm': lgbm, 'rf': rf, 'mlp': mlp_final,
                   'scaler': final_scaler, 'meta': meta_final, 'platt': platt}
    }
    return result

def predict_with_fitted_stack(fit, Xnew):
    Xnew = pd.DataFrame(Xnew, columns=fit['X_columns']).to_numpy(dtype=np.float32)
    m = fit['models']
    base = np.column_stack([
        m['xgb'].predict_proba(Xnew)[:,1], m['lgbm'].predict_proba(Xnew)[:,1],
        m['rf'].predict_proba(Xnew)[:,1], m['mlp'].predict(m['scaler'].transform(Xnew), verbose=0).ravel()
    ])
    raw = m['meta'].predict_proba(base)[:,1]
    cal = m['platt'].predict_proba(raw.reshape(-1,1))[:,1]
    return raw, cal

In [7]:
# Dataset preparation functions.
YESNO = {'No':0, 'Yes':1}
GENHEALTH = {'Excellent':1, 'Very good':2, 'Good':3, 'Fair':4, 'Poor':5}
AGE_ORDER = ['18-24','25-29','30-34','35-39','40-44','45-49','50-54','55-59','60-64','65-69','70-74','75-79','80 or older']
AGE_MAP = {v:i+1 for i,v in enumerate(AGE_ORDER)}
DIABETES_SEMANTIC = {'No':0, 'No, borderline diabetes':1, 'Yes (during pregnancy)':1, 'Yes':2}
RACE_CATS = ['American Indian/Alaskan Native','Asian','Black','Hispanic','Other','White']

SHARED15 = ['BMI','Smoker','Stroke','PhysHlth','MentHlth','DiffWalk','Sex','Age','Diabetes','PhysActivity','GenHlth']

def prepare_2015():
    df = pd.read_csv(DATA15)
    y = df['HeartDiseaseorAttack'].astype(int)
    X = df.drop(columns=['HeartDiseaseorAttack']).astype(float)
    groups = df[['Sex','Age','Income']].copy()
    return X, y, groups, df

def prepare_2020():
    df = pd.read_csv(DATA20)
    y = (df['HeartDisease'] == 'Yes').astype(int)
    X = pd.DataFrame(index=df.index)
    # Numeric variables.
    for c in ['BMI','PhysicalHealth','MentalHealth','SleepTime']:
        X[c] = pd.to_numeric(df[c], errors='raise')
    # Binary variables.
    for c in ['Smoking','AlcoholDrinking','Stroke','DiffWalking','PhysicalActivity','Asthma','KidneyDisease','SkinCancer']:
        X[c] = df[c].map(YESNO).astype(int)
    X['Sex'] = df['Sex'].map({'Female':0,'Male':1}).astype(int)
    X['AgeCategory'] = df['AgeCategory'].map(AGE_MAP).astype(int)
    X['Diabetic'] = df['Diabetic'].map(DIABETES_SEMANTIC).astype(int)
    X['GenHealth'] = df['GenHealth'].map(GENHEALTH).astype(int)
    # Fixed-category one-hot encoding for nominal race.
    for race in RACE_CATS:
        X['Race_' + race.replace(' ','_').replace('/','_')] = (df['Race'] == race).astype(int)
    groups = pd.DataFrame({'Sex':X['Sex'], 'AgeCategory':X['AgeCategory'], 'Race':df['Race']})
    return X, y, groups, df

def prepare_cardio():
    df = pd.read_csv(DATAC, sep=';').drop(columns=['id'])
    df['age'] = df['age'] / 365.25
    df['bmi'] = df['weight'] / (df['height']/100.0)**2
    df = df[(df.ap_hi>=60)&(df.ap_hi<=250)&(df.ap_lo>=40)&(df.ap_lo<=160)&
            (df.height>=100)&(df.height<=220)&(df.weight>=30)&(df.weight<=200)].reset_index(drop=True)
    cols = ['age','gender','height','weight','ap_hi','ap_lo','cholesterol','gluc','smoke','alco','active','bmi']
    X = df[cols].astype(float)
    y = df['cardio'].astype(int)
    groups = pd.DataFrame({'Gender':df['gender'].map({1:'Female',2:'Male'}), 'Age':df['age']})
    return X, y, groups, df

def prepare_2020_shared(df20):
    X = pd.DataFrame(index=df20.index)
    X['BMI'] = df20['BMI'].astype(float)
    X['Smoker'] = df20['Smoking'].map(YESNO).astype(int)
    X['Stroke'] = df20['Stroke'].map(YESNO).astype(int)
    X['PhysHlth'] = df20['PhysicalHealth'].astype(float)
    X['MentHlth'] = df20['MentalHealth'].astype(float)
    X['DiffWalk'] = df20['DiffWalking'].map(YESNO).astype(int)
    X['Sex'] = df20['Sex'].map({'Female':0,'Male':1}).astype(int)
    X['Age'] = df20['AgeCategory'].map(AGE_MAP).astype(int)
    X['Diabetes'] = df20['Diabetic'].map({'No':0, 'No, borderline diabetes':1, 'Yes (during pregnancy)':1, 'Yes':2}).astype(int)
    X['PhysActivity'] = df20['PhysicalActivity'].map(YESNO).astype(int)
    X['GenHlth'] = df20['GenHealth'].map(GENHEALTH).astype(int)
    return X[SHARED15]

def prepare_2020_shared_scrambled(df20):
    X = prepare_2020_shared(df20).copy()
    # Reproduce blind alphabetical label encoding only for the two disputed ordinal variables.
    X['GenHlth'] = pd.Categorical(df20['GenHealth'], categories=sorted(df20['GenHealth'].unique())).codes
    X['Diabetes'] = pd.Categorical(df20['Diabetic'], categories=sorted(df20['Diabetic'].unique())).codes
    return X

def align_2020_to_2015_full(df20, features15, medians15):
    shared = prepare_2020_shared(df20)
    X = pd.DataFrame(index=df20.index)
    for c in features15:
        if c in shared.columns:
            X[c] = shared[c]
        else:
            X[c] = float(medians15[c])
    return X[features15]

In [8]:
# File provenance checks used by the editor-response supplement.
def file_hash(path, alg='sha256'):
    h = hashlib.new(alg)
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1<<20), b''): h.update(chunk)
    return h.hexdigest()

provenance = {
    'BRFSS2015': {'file': os.path.basename(DATA15), 'md5': file_hash(DATA15,'md5'), 'sha256': file_hash(DATA15)},
    'BRFSS2020': {'file': os.path.basename(DATA20), 'md5': file_hash(DATA20,'md5'), 'sha256': file_hash(DATA20)},
    'Cardio': {'file': os.path.basename(DATAC), 'md5': file_hash(DATAC,'md5'), 'sha256': file_hash(DATAC)}
}
print(json.dumps(provenance, indent=2))
json.dump(provenance, open(os.path.join(OUT,'file_provenance.json'),'w'), indent=2)

{
  "BRFSS2015": {
    "file": "heart_disease_health_indicators_BRFSS2015.csv",
    "md5": "1044a4d01830fd9df016eb0d68f627b1",
    "sha256": "953fb23bb131736d7879539a071d3f6d1eaae8d4867ab05335d0ba89b94d5f99"
  },
  "BRFSS2020": {
    "file": "heart_2020_cleaned.csv",
    "md5": "f5a885ff39a113e797af67367b6bc2cf",
    "sha256": "eb1bfd667d199d19380119491d41d3f30b9868a8b1fa7c5bed12c3f048de5d49"
  },
  "Cardio": {
    "file": "cardio_train.csv",
    "md5": "71c46a70f7ba9e3e4736ae73ecd5bac9",
    "sha256": "21a705d23381b0dfd6a6416da701b490744f1fc3b47e9ff3db3968c420ffa10c"
  }
}


## Run all analysis arms

The notebook will save each fitted pipeline and prediction file. Do not interrupt after OOF folds have begun.

In [9]:
X15, y15, g15, df15 = prepare_2015()
FIT15 = fit_fullstack_crossfitted(X15, y15, 0.12, 'BRFSS 2015', g15)
print(FIT15['metrics_calibrated'])

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


BRFSS 2015: base OOF fold 1/5


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


BRFSS 2015: base OOF fold 2/5


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


...

BRFSS 2015: base OOF fold 4/5


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


BRFSS 2015: base OOF fold 5/5
BRFSS 2015: meta OOF fold 1/5
BRFSS 2015: meta OOF fold 2/5
BRFSS 2015: meta OOF fold 3/5
BRFSS 2015: meta OOF fold 4/5
BRFSS 2015: meta OOF fold 5/5


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


{'AUC': 0.8501870497200448, 'AUPRC': 0.3707198983019049, 'Brier': 0.0706671980935273, 'ECE': 0.011363734908841518, 'Sensitivity': 0.782381251307805, 'Specificity': 0.7633875144156494, 'FPR': 0.23661248558435058, 'PPV': 0.2558680626839116, 'NPV': 0.9712094787254658, 'SelectionRate': 0.2880203405865657, 'BalancedAccuracy': 0.7728843828617272, 'TN': 35083, 'FP': 10874, 'FN': 1040, 'TP': 3739, 'threshold': 0.12}


In [10]:
X20, y20, g20, df20 = prepare_2020()
FIT20 = fit_fullstack_crossfitted(X20, y20, 0.12, 'BRFSS 2020 retrained', g20)
print('Encoded 2020 design columns:', len(X20.columns))
print(FIT20['metrics_calibrated'])

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


BRFSS 2020 retrained: base OOF fold 1/5


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


BRFSS 2020 retrained: base OOF fold 2/5


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


...

BRFSS 2020 retrained: base OOF fold 4/5


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


BRFSS 2020 retrained: base OOF fold 5/5
BRFSS 2020 retrained: meta OOF fold 1/5
BRFSS 2020 retrained: meta OOF fold 2/5
BRFSS 2020 retrained: meta OOF fold 3/5
BRFSS 2020 retrained: meta OOF fold 4/5
BRFSS 2020 retrained: meta OOF fold 5/5


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Encoded 2020 design columns: 22
{'AUC': 0.8408839134553134, 'AUPRC': 0.3540703919580387, 'Brier': 0.0661080136696005, 'ECE': 0.011707872752130585, 'Sensitivity': 0.7358904109589041, 'Specificity': 0.7829662813761029, 'FPR': 0.21703371862389714, 'PPV': 0.24094007893792607, 'NPV': 0.969388403158541, 'SelectionRate': 0.2614487405994465, 'BalancedAccuracy': 0.7594283461675035, 'TN': 45791, 'FP': 12693, 'FN': 1446, 'TP': 4029, 'threshold': 0.12}


In [11]:
# Paired 2020 race-feature ablation using the same split and the same cross-fitting design.
RACE_COLUMNS = [c for c in X20.columns if c.startswith('Race_')]
X20_NO_RACE = X20.drop(columns=RACE_COLUMNS)
FIT20_NO_RACE = fit_fullstack_crossfitted(X20_NO_RACE, y20, 0.12, 'BRFSS 2020 without race', g20)
assert np.array_equal(FIT20['test_indices'], FIT20_NO_RACE['test_indices'])
print(FIT20_NO_RACE['metrics_calibrated'])

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


BRFSS 2020 without race: base OOF fold 1/5


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


BRFSS 2020 without race: base OOF fold 2/5


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


...

BRFSS 2020 without race: base OOF fold 4/5


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


BRFSS 2020 without race: base OOF fold 5/5
BRFSS 2020 without race: meta OOF fold 1/5
BRFSS 2020 without race: meta OOF fold 2/5
BRFSS 2020 without race: meta OOF fold 3/5
BRFSS 2020 without race: meta OOF fold 4/5
BRFSS 2020 without race: meta OOF fold 5/5


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


{'AUC': 0.8396830292576605, 'AUPRC': 0.35031742686946304, 'Brier': 0.06632100415396977, 'ECE': 0.012619686819440167, 'Sensitivity': 0.7404566210045662, 'Specificity': 0.7759729156692429, 'FPR': 0.22402708433075713, 'PPV': 0.23630216833760784, 'NPV': 0.9696386983740358, 'SelectionRate': 0.26823433762253945, 'BalancedAccuracy': 0.7582147683369045, 'TN': 45382, 'FP': 13102, 'FN': 1421, 'TP': 4054, 'threshold': 0.12}


In [12]:
Xc, yc, gc, dfc = prepare_cardio()
FITC = fit_fullstack_crossfitted(Xc, yc, 0.50, 'Cardio secondary benchmark', gc)
print(FITC['metrics_calibrated'])

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Cardio secondary benchmark: base OOF fold 1/5


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Cardio secondary benchmark: base OOF fold 2/5


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


...

Cardio secondary benchmark: base OOF fold 4/5


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Cardio secondary benchmark: base OOF fold 5/5
Cardio secondary benchmark: meta OOF fold 1/5
Cardio secondary benchmark: meta OOF fold 2/5
Cardio secondary benchmark: meta OOF fold 3/5
Cardio secondary benchmark: meta OOF fold 4/5
Cardio secondary benchmark: meta OOF fold 5/5


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


{'AUC': 0.8068023204411281, 'AUPRC': 0.7882016385200803, 'Brier': 0.17966394336504546, 'ECE': 0.029289604114879006, 'Sensitivity': 0.6845045574830932, 'Specificity': 0.7874423963133641, 'FPR': 0.21255760368663595, 'PPV': 0.7592954990215264, 'NPV': 0.7181507748883635, 'SelectionRate': 0.4460934089917067, 'BalancedAccuracy': 0.7359734768982287, 'TN': 5468, 'FP': 1476, 'FN': 2146, 'TP': 4656, 'threshold': 0.5}


In [13]:
# Naive 2015 -> 2020 transport with frozen full 2015 stack/calibrator.
med15 = FIT15['X_train'].median()
X20_full_aligned = align_2020_to_2015_full(df20, FIT15['X_columns'], med15)
raw20_transport, prob20_transport = predict_with_fitted_stack(FIT15, X20_full_aligned)
y20_all = (df20['HeartDisease'] == 'Yes').astype(int).to_numpy()
METRIC_TRANSPORT = metrics_at_threshold(y20_all, prob20_transport, 0.12)
print(METRIC_TRANSPORT)

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


{'AUC': 0.8393397777426091, 'AUPRC': 0.3354429166918969, 'Brier': 0.06847471383784041, 'ECE': 0.0331020879329038, 'Sensitivity': 0.5208417053300698, 'Specificity': 0.8914513955858314, 'FPR': 0.10854860441416857, 'PPV': 0.30994152046783624, 'NPV': 0.9520957209016933, 'SelectionRate': 0.14383902187338765, 'BalancedAccuracy': 0.7061465504579506, 'TN': 260680, 'FP': 31742, 'FN': 13116, 'TP': 14257, 'threshold': 0.12}


In [14]:
# Harmonised shared-feature model and transport.
X15_shared = X15[SHARED15].copy()
FIT15_SHARED = fit_fullstack_crossfitted(X15_shared, y15, 0.12, 'BRFSS 2015 shared-feature', g15)
X20_shared = prepare_2020_shared(df20)
raw20_shared, prob20_shared = predict_with_fitted_stack(FIT15_SHARED, X20_shared)
METRIC_SHARED20 = metrics_at_threshold(y20_all, prob20_shared, 0.12)
METRIC_SHARED15 = FIT15_SHARED['metrics_calibrated']

X20_scrambled = prepare_2020_shared_scrambled(df20)
_, prob20_scrambled = predict_with_fitted_stack(FIT15_SHARED, X20_scrambled)
METRIC_SCRAMBLED = metrics_at_threshold(y20_all, prob20_scrambled, 0.12)

print('shared 2015:', METRIC_SHARED15)
print('shared 2020:', METRIC_SHARED20)
print('encoding-scrambled 2020:', METRIC_SCRAMBLED)

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


BRFSS 2015 shared-feature: base OOF fold 1/5


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


BRFSS 2015 shared-feature: base OOF fold 2/5


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


...

BRFSS 2015 shared-feature: base OOF fold 4/5


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


BRFSS 2015 shared-feature: base OOF fold 5/5
BRFSS 2015 shared-feature: meta OOF fold 1/5
BRFSS 2015 shared-feature: meta OOF fold 2/5
BRFSS 2015 shared-feature: meta OOF fold 3/5
BRFSS 2015 shared-feature: meta OOF fold 4/5
BRFSS 2015 shared-feature: meta OOF fold 5/5


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


shared 2015: {'AUC': 0.8384732513520797, 'AUPRC': 0.35140909360790484, 'Brier': 0.07196557935375986, 'ECE': 0.010683075779973713, 'Sensitivity': 0.7679430843272651, 'Specificity': 0.7550536370955458, 'FPR': 0.24494636290445415, 'PPV': 0.2458632009111007, 'NPV': 0.9690301320896981, 'SelectionRate': 0.2942092399873857, 'BalancedAccuracy': 0.7614983607114054, 'TN': 34700, 'FP': 11257, 'FN': 1109, 'TP': 3670, 'threshold': 0.12}
shared 2020: {'AUC': 0.840540972089755, 'AUPRC': 0.3399001360338248, 'Brier': 0.066500314848856, 'ECE': 0.012437823553225542, 'Sensitivity': 0.7432506484492017, 'Specificity': 0.7774517649150885, 'FPR': 0.22254823508491153, 'PPV': 0.23816770658956019, 'NPV': 0.970013482839247, 'SelectionRate': 0.2671179974671274, 'BalancedAccuracy': 0.7603512066821452, 'TN': 227344, 'FP': 65078, 'FN': 7028, 'TP': 20345, 'threshold': 0.12}
encoding-scrambled 2020: {'AUC': 0.7670574549904039, 'AUPRC': 0.21421579677052865, 'Brier': 0.07420148191445941, 'ECE': 0.026940744525595032, 'Sen

In [15]:
# Bootstrap CI for the independent-sample sensitivity difference (2015 test minus all-2020 transport).
def bootstrap_sens_difference(y_a, p_a, y_b, p_b, threshold=0.12, B=1000, seed=SEED):
    rng = np.random.default_rng(seed)
    y_a=np.asarray(y_a); p_a=np.asarray(p_a); y_b=np.asarray(y_b); p_b=np.asarray(p_b)
    diffs=[]
    for _ in range(B):
        ia=rng.integers(0,len(y_a),len(y_a)); ib=rng.integers(0,len(y_b),len(y_b))
        sa=((p_a[ia]>=threshold)&(y_a[ia]==1)).sum()/max((y_a[ia]==1).sum(),1)
        sb=((p_b[ib]>=threshold)&(y_b[ib]==1)).sum()/max((y_b[ib]==1).sum(),1)
        diffs.append(sa-sb)
    return np.quantile(diffs,[0.025,0.975]).tolist()

shared_ci = bootstrap_sens_difference(
    FIT15_SHARED['y_test'], FIT15_SHARED['prob_test'], y20_all, prob20_shared
)
print('Shared sensitivity difference:', METRIC_SHARED15['Sensitivity']-METRIC_SHARED20['Sensitivity'], 'CI', shared_ci)

Shared sensitivity difference: 0.024692435878063357 CI [0.011208546288468286, 0.03781296445609697]


## Fairness and validation-fitted equal-opportunity threshold

The female threshold is estimated from cross-fitted development scores and then applied unchanged to the locked test set. This prevents the fairness mitigation from being optimised on the evaluation sample.

In [16]:
def binary_group_tpr(y, prob, group, threshold, positive_value):
    y=np.asarray(y); prob=np.asarray(prob); group=np.asarray(group)
    m=(group==positive_value)&(y==1)
    return float((prob[m]>=threshold).mean()) if m.sum() else np.nan

def fairness_binary(y, prob, group, threshold, values):
    out={}
    for v in values:
        m=np.asarray(group)==v
        yy=np.asarray(y)[m]; pp=np.asarray(prob)[m]
        met=metrics_at_threshold(yy,pp,threshold)
        out[str(v)]={k:met[k] for k in ['Sensitivity','Specificity','FPR','PPV','NPV','SelectionRate','Brier','ECE']}
    tprs=[out[str(v)]['Sensitivity'] for v in values]
    return {'groups':out,'TPR_gap':float(np.nanmax(tprs)-np.nanmin(tprs))}

def choose_equal_opportunity_threshold(y_dev, prob_dev, group_dev, male_threshold=0.12):
    y=np.asarray(y_dev); p=np.asarray(prob_dev); g=np.asarray(group_dev)
    male_tpr=binary_group_tpr(y,p,g,male_threshold,1)
    candidates=np.unique(p[g==0])
    female_positive=(g==0)&(y==1)
    tprs=np.array([(p[female_positive]>=t).mean() for t in candidates])
    idx=np.argmin(np.abs(tprs-male_tpr))
    return float(candidates[idx]), float(male_tpr), float(tprs[idx])

# Development groups follow the same split used by FIT15.
g15_train = FIT15['groups_train']
g15_test  = FIT15['groups_test']
female_threshold, male_dev_tpr, female_dev_tpr = choose_equal_opportunity_threshold(
    FIT15['y_train'], FIT15['prob_dev_cf'], g15_train['Sex'].to_numpy(), 0.12
)

fair_sex_pre = fairness_binary(FIT15['y_test'], FIT15['prob_test'], g15_test['Sex'], 0.12, [0,1])
# Apply group-specific thresholds on locked test.
yte=FIT15['y_test'].to_numpy(); pte=FIT15['prob_test']; sexte=g15_test['Sex'].to_numpy()
pred_post=np.where(sexte==0, pte>=female_threshold, pte>=0.12).astype(int)

def metrics_from_binary(y,pred):
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    sens=tp/(tp+fn); spec=tn/(tn+fp)
    return {'Sensitivity':sens,'Specificity':spec,'FPR':fp/(fp+tn),'PPV':tp/(tp+fp),
            'NPV':tn/(tn+fn),'SelectionRate':pred.mean(),'BalancedAccuracy':0.5*(sens+spec)}
post_overall=metrics_from_binary(yte,pred_post)
f_tpr=pred_post[(sexte==0)&(yte==1)].mean(); m_tpr=pred_post[(sexte==1)&(yte==1)].mean()
mitigation={'female_threshold':female_threshold,'male_threshold':0.12,'female_test_TPR':f_tpr,'male_test_TPR':m_tpr,
            'test_TPR_gap':abs(f_tpr-m_tpr),'overall':post_overall,
            'net_benefit': ((pred_post & (yte==1)).sum()/len(yte)) - ((pred_post & (yte==0)).sum()/len(yte))*(0.12/(1-0.12))}

fair_cardio = fairness_binary(FITC['y_test'], FITC['prob_test'], FITC['groups_test']['Gender'], 0.50, ['Female','Male'])
fair_2020_sex = fairness_binary(FIT20['y_test'], FIT20['prob_test'], FIT20['groups_test']['Sex'], 0.12, [0,1])

print('sex fairness pre:', fair_sex_pre)
print('mitigation selected on development:', mitigation)
print('2020 sex fairness:', fair_2020_sex)
print('Cardio gender fairness:', fair_cardio)

# Multigroup and intersectional summaries used in Table 5.
def fairness_multigroup(y, prob, group, threshold, min_positive=1):
    y=np.asarray(y); p=np.asarray(prob); g=np.asarray(group)
    rows=[]
    for value in pd.unique(g):
        m=(g==value); pos=(m & (y==1)).sum()
        if pos < min_positive: continue
        pred=(p[m]>=threshold).astype(int); yy=y[m]
        met=metrics_from_binary(yy,pred)
        rows.append({'group':str(value),'n':int(m.sum()),'positives':int(pos),'TPR':float(met['Sensitivity']),
                     'selection_rate':float(met['SelectionRate'])})
    gap=float(max(r['TPR'] for r in rows)-min(r['TPR'] for r in rows)) if len(rows)>=2 else np.nan
    return {'gap':gap,'groups':rows}

def fairness_intersection(y, prob, g1, g2, threshold, min_positive=30):
    labels=np.array([f'{a}|{b}' for a,b in zip(g1,g2)],dtype=object)
    return fairness_multigroup(y,prob,labels,threshold,min_positive)

# 2015 locked test.
fair_2015_age = fairness_multigroup(yte,pte,g15_test['Age'],0.12)
fair_2015_income = fairness_multigroup(yte,pte,g15_test['Income'],0.12)
fair_2015_sex_age = fairness_intersection(yte,pte,g15_test['Sex'],g15_test['Age'],0.12,30)

# Naive 2020 transport on the complete 2020 benchmark.
g20_all=pd.DataFrame({'Sex':df20['Sex'].map({'Female':0,'Male':1}),
                      'AgeCategory':df20['AgeCategory'].map(AGE_MAP),
                      'Race':df20['Race']})
fair_transport_sex=fairness_binary(y20_all,prob20_transport,g20_all['Sex'],0.12,[0,1])
fair_transport_age=fairness_multigroup(y20_all,prob20_transport,g20_all['AgeCategory'],0.12)
fair_transport_race=fairness_multigroup(y20_all,prob20_transport,g20_all['Race'],0.12)

# Retrained 2020 locked test.
y20te=FIT20['y_test'].to_numpy(); p20te=FIT20['prob_test']; g20te=FIT20['groups_test']
fair_2020_age=fairness_multigroup(y20te,p20te,g20te['AgeCategory'],0.12)
fair_2020_race=fairness_multigroup(y20te,p20te,g20te['Race'],0.12)

# Cardio age bands matching the original notebook.
cardio_age_group=pd.cut(FITC['X_test']['age'],bins=[0,35,45,55,65,100],labels=[1,2,3,4,5]).astype(float)
fair_cardio_age=fairness_multigroup(FITC['y_test'],FITC['prob_test'],cardio_age_group,0.50)

print('2015 age gap',fair_2015_age['gap'],'income gap',fair_2015_income['gap'],'sex-age max',fair_2015_sex_age['gap'])
print('transport age/race gaps',fair_transport_age['gap'],fair_transport_race['gap'])
print('2020 age/race gaps',fair_2020_age['gap'],fair_2020_race['gap'])
print('Cardio age gap',fair_cardio_age['gap'])


# Bootstrap uncertainty and paired race-feature ablation.
auc15_ci=bootstrap_auc_ci(FIT15['y_test'],FIT15['prob_test'])
sex_gap15_ci=bootstrap_binary_gap_ci(yte,pte,g15_test['Sex'],0.12,[0,1])

def bootstrap_intersection_gap_ci(y,prob,g1,g2,threshold,min_positive=30,B=1000,seed=SEED):
    rng=np.random.default_rng(seed); y=np.asarray(y); prob=np.asarray(prob); g1=np.asarray(g1); g2=np.asarray(g2); vals=[]
    n=len(y)
    for _ in range(B):
        idx=rng.integers(0,n,n)
        res=fairness_intersection(y[idx],prob[idx],g1[idx],g2[idx],threshold,min_positive)
        if np.isfinite(res['gap']): vals.append(res['gap'])
    return np.quantile(vals,[0.025,0.975]).tolist()

sex_age_ci=bootstrap_intersection_gap_ci(yte,pte,g15_test['Sex'],g15_test['Age'],0.12,30)

# Paired bootstrap difference in race TPR gap: with race minus without race.
y20te2=FIT20_NO_RACE['y_test'].to_numpy(); p20nr=FIT20_NO_RACE['prob_test']; race20=g20te['Race'].to_numpy()
def race_gap(y,p,r): return fairness_multigroup(y,p,r,0.12)['gap']
rng=np.random.default_rng(SEED); diffs=[]
for _ in range(1000):
    idx=rng.integers(0,len(y20te),len(y20te))
    diffs.append(race_gap(y20te[idx],p20te[idx],race20[idx])-race_gap(y20te2[idx],p20nr[idx],race20[idx]))
race_with_gap=fair_2020_race['gap']; race_without_gap=race_gap(y20te2,p20nr,race20)
race_diff=race_with_gap-race_without_gap; race_diff_ci=np.quantile(diffs,[0.025,0.975]).tolist()

# Threshold-sensitivity summaries.
thresholds=np.linspace(0.05,0.30,26)
sex_gap_curve=[]; naive_loss_curve=[]; shared_loss_curve=[]
for t in thresholds:
    sex_gap_curve.append(fairness_binary(yte,pte,g15_test['Sex'],t,[0,1])['TPR_gap'])
    s15=metrics_at_threshold(FIT15['y_test'],FIT15['prob_test'],t)['Sensitivity']
    s20=metrics_at_threshold(y20_all,prob20_transport,t)['Sensitivity']
    hs15=metrics_at_threshold(FIT15_SHARED['y_test'],FIT15_SHARED['prob_test'],t)['Sensitivity']
    hs20=metrics_at_threshold(y20_all,prob20_shared,t)['Sensitivity']
    naive_loss_curve.append(s15-s20); shared_loss_curve.append(hs15-hs20)
threshold_sensitivity={'thresholds':thresholds.tolist(),'sex_gap':sex_gap_curve,
                       'sex_gap_range':[float(np.min(sex_gap_curve)),float(np.max(sex_gap_curve))],
                       'naive_loss_range':[float(np.min(naive_loss_curve)),float(np.max(naive_loss_curve))],
                       'shared_loss_range':[float(np.min(shared_loss_curve)),float(np.max(shared_loss_curve))]}
print('AUC CI',auc15_ci,'sex gap CI',sex_gap15_ci,'sex-age CI',sex_age_ci)
print('race ablation',race_with_gap,race_without_gap,race_diff,race_diff_ci)


sex fairness pre: {'groups': {'0': {'Sensitivity': 0.711847879083374, 'Specificity': 0.8172501809592746, 'FPR': 0.18274981904072535, 'PPV': 0.233338660700016, 'NPV': 0.9731887674091548, 'SelectionRate': 0.22109540636042402, 'Brier': 0.057156086786285254, 'ECE': 0.009582048269936897}, '1': {'Sensitivity': 0.8354105571847508, 'Specificity': 0.6916480617008322, 'FPR': 0.30835193829916785, 'PPV': 0.27273815222594544, 'NPV': 0.9681107954545455, 'SelectionRate': 0.3724371545730077, 'Brier': 0.08770964995637939, 'ECE': 0.013611093256185114}}, 'TPR_gap': 0.12356267810137678}
mitigation selected on development: {'female_threshold': 0.05302095416216619, 'male_threshold': 0.12, 'female_test_TPR': np.float64(0.8352023403217943), 'male_test_TPR': np.float64(0.8354105571847508), 'test_TPR_gap': np.float64(0.00020821686295646735), 'overall': {'Sensitivity': np.float64(0.8353211969031178), 'Specificity': np.float64(0.7105772787605805), 'FPR': np.float64(0.2894227212394195), 'PPV': np.float64(0.2308448

/tmp/ipykernel_1156/2084564836.py:40: RuntimeWarning: invalid value encountered in scalar divide
  return {'Sensitivity':sens,'Specificity':spec,'FPR':fp/(fp+tn),'PPV':tp/(tp+fp),


2015 age gap 0.9095238095238095 income gap 0.22833983866770746 sex-age max 0.6560415122312825
transport age/race gaps 0.7487612405946045 0.11671691560990455
2020 age/race gaps 0.8680659173049505 0.2003703879953047
Cardio age gap 0.21187570886620521
AUC CI [0.8447339099382332, 0.854924228696898] sex gap CI [0.09964134745823332, 0.14868793415703868] sex-age CI [0.542853554094476, 0.9059458472418477]
race ablation 0.2003703879953047 0.16475155279503106 0.035618835200273646 [-0.017317320626634956, 0.11112397361575262]


In [17]:
# Save models and predictions.
def save_fit(fit, slug):
    d=os.path.join(MODEL_OUT,slug); os.makedirs(d,exist_ok=True)
    for key in ['xgb','lgbm','rf','scaler','meta','platt']:
        joblib.dump(fit['models'][key], os.path.join(d,key+'.pkl'))
    fit['models']['mlp'].save(os.path.join(d,'mlp.keras'))
    pred=pd.DataFrame({'y_true':fit['y_test'].to_numpy(),'raw_stack':fit['raw_test'],'prob_calibrated':fit['prob_test']})
    if fit['groups_test'] is not None:
        for c in fit['groups_test'].columns: pred[c]=fit['groups_test'][c].to_numpy()
    pred.to_csv(os.path.join(OUT,slug+'_test_predictions.csv'),index=False)

save_fit(FIT15,'brfss2015'); save_fit(FIT20,'brfss2020'); save_fit(FITC,'cardio'); save_fit(FIT15_SHARED,'brfss2015_shared')
transport_pred = pd.DataFrame({
    'y_true': y20_all, 'raw_stack': raw20_transport, 'prob_calibrated': prob20_transport,
    'Sex': g20_all['Sex'].to_numpy(), 'AgeCategory': g20_all['AgeCategory'].to_numpy(),
    'Race': g20_all['Race'].to_numpy()
})
transport_pred.to_csv(os.path.join(OUT,'brfss2020_transport_predictions.csv'),index=False)
shared_pred = pd.DataFrame({
    'y_true': y20_all, 'raw_stack': raw20_shared, 'prob_calibrated': prob20_shared,
    'Sex': g20_all['Sex'].to_numpy(), 'AgeCategory': g20_all['AgeCategory'].to_numpy(),
    'Race': g20_all['Race'].to_numpy()
})
shared_pred.to_csv(os.path.join(OUT,'brfss2020_shared_transport_predictions.csv'),index=False)

In [18]:
# Master metrics and manuscript update map.
def rounded(d, n=4):
    out={}
    for k,v in d.items():
        if isinstance(v,(float,np.floating)): out[k]=round(float(v),n)
        elif isinstance(v,(int,np.integer)): out[k]=int(v)
        else: out[k]=v
    return out

master={
    'method': 'two-stage full-stack cross-fitting: base OOF -> meta OOF -> Platt',
    'BRFSS2015': rounded(FIT15['metrics_calibrated']),
    'BRFSS2015_raw': rounded(FIT15['metrics_raw']),
    'BRFSS2020_retrained': rounded(FIT20['metrics_calibrated']),
    'BRFSS2020_retrained_raw': rounded(FIT20['metrics_raw']),
    'Cardio': rounded(FITC['metrics_calibrated']),
    'Cardio_raw': rounded(FITC['metrics_raw']),
    'BRFSS2020_naive_transport': rounded(METRIC_TRANSPORT),
    'BRFSS2015_shared': rounded(METRIC_SHARED15),
    'BRFSS2020_shared_transport': rounded(METRIC_SHARED20),
    'BRFSS2020_shared_scrambled': rounded(METRIC_SCRAMBLED),
    'shared_sensitivity_difference': round(METRIC_SHARED15['Sensitivity']-METRIC_SHARED20['Sensitivity'],4),
    'shared_sensitivity_difference_CI95': [round(x,4) for x in shared_ci],
    'BRFSS2015_sex_fairness': fair_sex_pre,
    'BRFSS2015_age_fairness': fair_2015_age,
    'BRFSS2015_income_fairness': fair_2015_income,
    'BRFSS2015_sex_age_intersection': fair_2015_sex_age,
    'BRFSS2015_equal_opportunity_mitigation': mitigation,
    'BRFSS2020_naive_transport_sex_fairness': fair_transport_sex,
    'BRFSS2020_naive_transport_age_fairness': fair_transport_age,
    'BRFSS2020_naive_transport_race_fairness': fair_transport_race,
    'BRFSS2020_sex_fairness': fair_2020_sex,
    'BRFSS2020_age_fairness': fair_2020_age,
    'BRFSS2020_race_fairness': fair_2020_race,
    'Cardio_gender_fairness': fair_cardio,
    'Cardio_age_fairness': fair_cardio_age,
    'provenance': provenance,
    'encoded_2020_design_columns': len(X20.columns),
    'BRFSS2015_AUC_CI95': [round(x,4) for x in auc15_ci],
    'BRFSS2015_sex_gap_CI95': [round(x,4) for x in sex_gap15_ci],
    'BRFSS2015_sex_age_gap_CI95': [round(x,4) for x in sex_age_ci],
    'BRFSS2020_race_ablation': {'with_race_gap':race_with_gap,'without_race_gap':race_without_gap,'difference':race_diff,'difference_CI95':race_diff_ci},
    'threshold_sensitivity': threshold_sensitivity
}
json.dump(master, open(os.path.join(OUT,'editor_revision_metrics.json'),'w'), indent=2)

update_map={
    'ENCODED20_COLS': master['encoded_2020_design_columns'],
    'BRFSS15_ECE_REDUCTION_PCT': 100.0*(master['BRFSS2015_raw']['ECE']-master['BRFSS2015']['ECE'])/master['BRFSS2015_raw']['ECE'],
    'RESPONSE_MAX_FAIRNESS_CHANGE': None,
    'BRFSS15_AUC': master['BRFSS2015']['AUC'],
    'BRFSS15_AUPRC': master['BRFSS2015']['AUPRC'],
    'BRFSS15_BRIER': master['BRFSS2015']['Brier'],
    'BRFSS15_ECE_RAW': master['BRFSS2015_raw']['ECE'],
    'BRFSS15_ECE': master['BRFSS2015']['ECE'],
    'BRFSS15_SENS': master['BRFSS2015']['Sensitivity'],
    'BRFSS15_SPEC': master['BRFSS2015']['Specificity'],
    'BRFSS15_FPR': master['BRFSS2015']['FPR'],
    'BRFSS15_PPV': master['BRFSS2015']['PPV'],
    'BRFSS15_NPV': master['BRFSS2015']['NPV'],
    'BRFSS15_SELECTION': master['BRFSS2015']['SelectionRate'],
    'BRFSS15_BALACC': master['BRFSS2015']['BalancedAccuracy'],
    'TRANSPORT_AUC': master['BRFSS2020_naive_transport']['AUC'],
    'TRANSPORT_AUPRC': master['BRFSS2020_naive_transport']['AUPRC'],
    'TRANSPORT_BRIER': master['BRFSS2020_naive_transport']['Brier'],
    'TRANSPORT_ECE': master['BRFSS2020_naive_transport']['ECE'],
    'TRANSPORT_SENS': master['BRFSS2020_naive_transport']['Sensitivity'],
    'TRANSPORT_SPEC': master['BRFSS2020_naive_transport']['Specificity'],
    'RETRAIN20_AUC': master['BRFSS2020_retrained']['AUC'],
    'RETRAIN20_AUPRC': master['BRFSS2020_retrained']['AUPRC'],
    'RETRAIN20_BRIER': master['BRFSS2020_retrained']['Brier'],
    'RETRAIN20_ECE_RAW': master['BRFSS2020_retrained_raw']['ECE'] if 'BRFSS2020_retrained_raw' in master else FIT20['metrics_raw']['ECE'],
    'RETRAIN20_ECE': master['BRFSS2020_retrained']['ECE'],
    'RETRAIN20_SENS': master['BRFSS2020_retrained']['Sensitivity'],
    'RETRAIN20_SPEC': master['BRFSS2020_retrained']['Specificity'],
    'CARDIO_AUC': master['Cardio']['AUC'],
    'CARDIO_AUPRC': master['Cardio']['AUPRC'],
    'CARDIO_BRIER': master['Cardio']['Brier'],
    'CARDIO_ECE_RAW': master['Cardio_raw']['ECE'] if 'Cardio_raw' in master else FITC['metrics_raw']['ECE'],
    'CARDIO_ECE': master['Cardio']['ECE'],
    'CARDIO_SENS': master['Cardio']['Sensitivity'],
    'CARDIO_SPEC': master['Cardio']['Specificity'],
    'SHARED15_AUC': master['BRFSS2015_shared']['AUC'],
    'SHARED15_ECE': master['BRFSS2015_shared']['ECE'],
    'SHARED15_SENS': master['BRFSS2015_shared']['Sensitivity'],
    'SHARED20_AUC': master['BRFSS2020_shared_transport']['AUC'],
    'SHARED20_ECE': master['BRFSS2020_shared_transport']['ECE'],
    'SHARED20_SENS': master['BRFSS2020_shared_transport']['Sensitivity'],
    'SHARED_DIFF': master['shared_sensitivity_difference'],
    'SHARED_CI_LOW': master['shared_sensitivity_difference_CI95'][0],
    'SHARED_CI_HIGH': master['shared_sensitivity_difference_CI95'][1],
    'SCRAMBLED_SENS': master['BRFSS2020_shared_scrambled']['Sensitivity'],
    'SEX_GAP_2015': fair_sex_pre['TPR_gap'],
    'FEMALE_TPR_2015': fair_sex_pre['groups']['0']['Sensitivity'],
    'MALE_TPR_2015': fair_sex_pre['groups']['1']['Sensitivity'],
    'MITIGATION_FEMALE_THRESHOLD': mitigation['female_threshold'],
    'MITIGATION_FEMALE_TPR': mitigation['female_test_TPR'],
    'MITIGATION_MALE_TPR': mitigation['male_test_TPR'],
    'MITIGATION_TEST_GAP': mitigation['test_TPR_gap'],
    'MITIGATION_SENS': mitigation['overall']['Sensitivity'],
    'MITIGATION_SPEC': mitigation['overall']['Specificity'],
    'MITIGATION_FPR': mitigation['overall']['FPR'],
    'MITIGATION_PPV': mitigation['overall']['PPV'],
    'MITIGATION_SELECTION': mitigation['overall']['SelectionRate'],
    'MITIGATION_BALACC': mitigation['overall']['BalancedAccuracy'],
    'MITIGATION_NET_BENEFIT': mitigation['net_benefit'],
    'CARDIO_SEX_GAP': fair_cardio['TPR_gap'],
    'CARDIO_FEMALE_TPR': fair_cardio['groups']['Female']['Sensitivity'],
    'CARDIO_MALE_TPR': fair_cardio['groups']['Male']['Sensitivity'],
    'SEX_GAP_2020': fair_2020_sex['TPR_gap'],
    'FEMALE_TPR_2020': fair_2020_sex['groups']['0']['Sensitivity'],
    'MALE_TPR_2020': fair_2020_sex['groups']['1']['Sensitivity'],
    'AGE_GAP_2015': fair_2015_age['gap'],
    'INCOME_GAP_2015': fair_2015_income['gap'],
    'SEX_AGE_MAX_2015': fair_2015_sex_age['gap'],
    'TRANSPORT_SEX_GAP': fair_transport_sex['TPR_gap'],
    'TRANSPORT_FEMALE_TPR': fair_transport_sex['groups']['0']['Sensitivity'],
    'TRANSPORT_MALE_TPR': fair_transport_sex['groups']['1']['Sensitivity'],
    'TRANSPORT_AGE_GAP': fair_transport_age['gap'],
    'TRANSPORT_RACE_GAP': fair_transport_race['gap'],
    'AGE_GAP_2020': fair_2020_age['gap'],
    'RACE_GAP_2020': fair_2020_race['gap'],
    'CARDIO_AGE_GAP': fair_cardio_age['gap'],
    'AUC15_CI_LOW': auc15_ci[0],
    'AUC15_CI_HIGH': auc15_ci[1],
    'SEX_GAP15_CI_LOW': sex_gap15_ci[0],
    'SEX_GAP15_CI_HIGH': sex_gap15_ci[1],
    'SEX_AGE_CI_LOW': sex_age_ci[0],
    'SEX_AGE_CI_HIGH': sex_age_ci[1],
    'RACE_GAP_WITH': race_with_gap,
    'RACE_GAP_WITHOUT': race_without_gap,
    'RACE_GAP_DIFF': race_diff,
    'RACE_GAP_DIFF_CI_LOW': race_diff_ci[0],
    'RACE_GAP_DIFF_CI_HIGH': race_diff_ci[1],
    'SEX_GAP_THRESHOLD_MIN': threshold_sensitivity['sex_gap_range'][0],
    'SEX_GAP_THRESHOLD_MAX': threshold_sensitivity['sex_gap_range'][1],
    'NAIVE_LOSS_THRESHOLD_MIN': threshold_sensitivity['naive_loss_range'][0],
    'NAIVE_LOSS_THRESHOLD_MAX': threshold_sensitivity['naive_loss_range'][1],
    'SHARED_LOSS_THRESHOLD_MIN': threshold_sensitivity['shared_loss_range'][0],
    'SHARED_LOSS_THRESHOLD_MAX': threshold_sensitivity['shared_loss_range'][1]
}

# Largest absolute change against the submitted calibration-dependent values (used only for the response letter).
submitted_reference = {
    'BRFSS15_SENS': 0.776, 'BRFSS15_SPEC': 0.769,
    'TRANSPORT_SENS': 0.267, 'RETRAIN20_SENS': 0.731,
    'CARDIO_SENS': 0.687, 'CARDIO_SPEC': 0.785,
    'BRFSS15_ECE': 0.011, 'RETRAIN20_ECE': 0.012, 'CARDIO_ECE': 0.027
}
changes=[]
for key, old in submitted_reference.items():
    if key in update_map and update_map[key] is not None:
        changes.append(abs(float(update_map[key])-old))
update_map['RESPONSE_MAX_DISPLAYED_CHANGE'] = max(changes) if changes else None
update_map.pop('RESPONSE_MAX_FAIRNESS_CHANGE', None)

json.dump(update_map, open(os.path.join(OUT,'manuscript_update_map.json'),'w'), indent=2)
print(json.dumps(master, indent=2)[:12000])

run_manifest = {
    'notebook': 'EXHEART_07_full_stack_crossfitted_calibration.ipynb',
    'method': master['method'],
    'seed': SEED,
    'base_folds': N_BASE_FOLDS,
    'meta_folds': N_META_FOLDS,
    'test_fraction': 0.20,
    'full_run': FULL_RUN,
    'fast_smoke_test': FAST_SMOKE_TEST,
    'output_files': sorted(os.listdir(OUT))
}
json.dump(run_manifest, open(os.path.join(OUT,'rerun_manifest.json'),'w'), indent=2)


{
  "method": "two-stage full-stack cross-fitting: base OOF -> meta OOF -> Platt",
  "BRFSS2015": {
    "AUC": 0.8502,
    "AUPRC": 0.3707,
    "Brier": 0.0707,
    "ECE": 0.0114,
    "Sensitivity": 0.7824,
    "Specificity": 0.7634,
    "FPR": 0.2366,
    "PPV": 0.2559,
    "NPV": 0.9712,
    "SelectionRate": 0.288,
    "BalancedAccuracy": 0.7729,
    "TN": 35083,
    "FP": 10874,
    "FN": 1040,
    "TP": 3739,
    "threshold": 0.12
  },
  "BRFSS2015_raw": {
    "AUC": 0.8502,
    "AUPRC": 0.3707,
    "Brier": 0.1688,
    "ECE": 0.2555,
    "Sensitivity": 0.973,
    "Specificity": 0.391,
    "FPR": 0.609,
    "PPV": 0.1425,
    "NPV": 0.9929,
    "SelectionRate": 0.6433,
    "BalancedAccuracy": 0.682,
    "TN": 17970,
    "FP": 27987,
    "FN": 129,
    "TP": 4650,
    "threshold": 0.12
  },
  "BRFSS2020_retrained": {
    "AUC": 0.8409,
    "AUPRC": 0.3541,
    "Brier": 0.0661,
    "ECE": 0.0117,
    "Sensitivity": 0.7359,
    "Specificity": 0.783,
    "FPR": 0.217,
    "PPV": 0.2409

## Consistency checks and final instructions

The cell below fails if the calibration design is not genuinely full-stack cross-fitted. After a successful full run, download:

- `editor_revision_metrics.json`
- `manuscript_update_map.json`
- all four test-prediction CSVs
- the revised model directories

Upload those files back to ChatGPT for automatic final insertion into the revised manuscript, response letter, figures, tables and Online Resource 1.

In [19]:
# Hard checks before using outputs.
assert FIT15['metrics_calibrated']['AUC'] == FIT15['metrics_raw']['AUC'] or abs(FIT15['metrics_calibrated']['AUC']-FIT15['metrics_raw']['AUC']) < 1e-10
assert FIT20['metrics_calibrated']['AUC'] == FIT20['metrics_raw']['AUC'] or abs(FIT20['metrics_calibrated']['AUC']-FIT20['metrics_raw']['AUC']) < 1e-10
assert FITC['metrics_calibrated']['AUC'] == FITC['metrics_raw']['AUC'] or abs(FITC['metrics_calibrated']['AUC']-FITC['metrics_raw']['AUC']) < 1e-10
assert len(FIT15['meta_cf_raw']) == len(FIT15['y_train'])
assert np.isfinite(FIT15['meta_cf_raw']).all()
print('PASS: full-stack cross-fitting and locked-test checks completed.')
print('Results folder:', OUT)
assert FIT15['groups_train'] is not None and len(FIT15['groups_train']) == len(FIT15['y_train'])
assert FIT15['groups_test'] is not None and len(FIT15['groups_test']) == len(FIT15['y_test'])
assert 'ENCODED20_COLS' in update_map
print('PASS: manuscript update map and grouped prediction exports are complete.')


PASS: full-stack cross-fitting and locked-test checks completed.
Results folder: /content/drive/MyDrive/EXHEART_Research/exheart-research/results/editor_revision_fullstack_crossfit
PASS: manuscript update map and grouped prediction exports are complete.


In [20]:
%cd /content/drive/MyDrive/EXHEART_Research/exheart-research
!git add results/editor_revision_fullstack_crossfit
!git commit -m "Editor revision: two-stage full-stack cross-fitted rerun (all arms)"
!git push

/content/drive/MyDrive/EXHEART_Research/exheart-research
[main e31eb94] Editor revision: two-stage full-stack cross-fitted rerun (all arms)
 10 files changed, 819914 insertions(+)
 create mode 100644 results/editor_revision_fullstack_crossfit/brfss2015_shared_test_predictions.csv
 create mode 100644 results/editor_revision_fullstack_crossfit/brfss2015_test_predictions.csv
 create mode 100644 results/editor_revision_fullstack_crossfit/brfss2020_shared_transport_predictions.csv
 create mode 100644 results/editor_revision_fullstack_crossfit/brfss2020_test_predictions.csv
 create mode 100644 results/editor_revision_fullstack_crossfit/brfss2020_transport_predictions.csv
 create mode 100644 results/editor_revision_fullstack_crossfit/cardio_test_predictions.csv
 create mode 100644 results/editor_revision_fullstack_crossfit/editor_revision_metrics.json
 create mode 100644 results/editor_revision_fullstack_crossfit/file_provenance.json
 create mode 100644 results/editor_revision_fullstack_cross

In [21]:
%cd /content/drive/MyDrive/EXHEART_Research/exheart-research
!git add -f models/editor_revision_fullstack_crossfit/brfss2015/meta.pkl models/editor_revision_fullstack_crossfit/brfss2020/meta.pkl models/editor_revision_fullstack_crossfit/cardio/meta.pkl
!git commit -m "Add cross-fitted meta-learner objects for figure regeneration"
!git push

/content/drive/MyDrive/EXHEART_Research/exheart-research
[main 6ee7b87] Add cross-fitted meta-learner objects for figure regeneration
 3 files changed, 0 insertions(+), 0 deletions(-)
 create mode 100644 models/editor_revision_fullstack_crossfit/brfss2015/meta.pkl
 create mode 100644 models/editor_revision_fullstack_crossfit/brfss2020/meta.pkl
 create mode 100644 models/editor_revision_fullstack_crossfit/cardio/meta.pkl
Enumerating objects: 12, done.
Counting objects: 100% (12/12), done.
Delta compression using up to 8 threads
Compressing objects: 100% (7/7), done.
Writing objects: 100% (10/10), 1.40 KiB | 142.00 KiB/s, done.
Total 10 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 1 local object.
To https://github.com/anasbiswas1/exheart-research.git
   e31eb94..6ee7b87  main -> main
